# GraphSAGE Extrapolation Training

This notebook trains GraphSAGE models on mixed-distance datasets (d=3, d=5, d=7) to study extrapolation to larger distances (d=9, d=11).

**Workflow:**
1. Load best hyperparameters from tuning results
2. Configure split experiments (different ratios of d=3/d=5/d=7 data)
3. Generate/load combined training datasets
4. Train models for each split configuration
5. Save trained models for evaluation in testing.ipynb

**Split Experiments (15 total):**

The experiments are organized into 4 groups to systematically study the effect of d=3 proportion and d5:d7 ratio:

**Control:**
- `equal_333333`: Baseline with equal distribution (33% d=3, 33% d=5, 34% d=7)

**Experiment A: Vary d=3 amount (d5:d7 = 1:1)**
- `a1_d3_00`: 0% d=3, 50% d=5, 50% d=7
- `a2_d3_10`: 10% d=3, 45% d=5, 45% d=7
- `a3_d3_20`: 20% d=3, 40% d=5, 40% d=7
- `a4_d3_40`: 40% d=3, 30% d=5, 30% d=7
- `a5_d3_50`: 50% d=3, 25% d=5, 25% d=7

**Experiment B: Vary d5:d7 ratio (no d=3)**
- `b1_d5heavy`: 0% d=3, 80% d=5, 20% d=7
- `b2_d5more`: 0% d=3, 65% d=5, 35% d=7
- `b3_balanced`: 0% d=3, 50% d=5, 50% d=7
- `b4_d7more`: 0% d=3, 35% d=5, 65% d=7
- `b5_d7heavy`: 0% d=3, 20% d=5, 80% d=7

**Experiment C: Extreme/boundary cases**
- `c1_only_d3`: 100% d=3, 0% d=5, 0% d=7
- `c2_only_d5`: 0% d=3, 100% d=5, 0% d=7
- `c3_only_d7`: 0% d=3, 0% d=5, 100% d=7
- `c4_no_d7`: 50% d=3, 50% d=5, 0% d=7

**Worker System:**
Experiments are divided into 3 workers (5 experiments each):
- **Worker 1**: equal_333333, a1_d3_00, a2_d3_10, a3_d3_20, a4_d3_40
- **Worker 2**: a5_d3_50, b1_d5heavy, b2_d5more, b3_balanced, b4_d7more
- **Worker 3**: b5_d7heavy, c1_only_d3, c2_only_d5, c3_only_d7, c4_no_d7

## Imports

In [ ]:
!pip install stim pymatching numpy matplotlib torch tqdm networkx
!pip install torch_geometric

In [ ]:
import sys
import json
import random
import time
from pathlib import Path
from datetime import datetime

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/quantum-error-correction/code')
else:
    BASE_PATH = Path('../..')  # code/gSAGE/extrapolation -> code/

sys.path.insert(0, str(BASE_PATH))

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from torch_geometric.loader import DataLoader

# Import from models.py
from models import (
    SurfaceCodeSampler,
    SparseGraph,
    GraphSAGEModel,
    GraphSAGE,
    DatasetCache,
)

# Set up paths
EXTRAPOLATION_DIR = BASE_PATH / "gSAGE" / "extrapolation"
EXTRAPOLATION_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = EXTRAPOLATION_DIR / "results" / "revised_training"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = EXTRAPOLATION_DIR / "models" / "revised_training"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = EXTRAPOLATION_DIR / "plots" / "revised_training"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"\nPaths:")
print(f"  BASE_PATH: {BASE_PATH}")
print(f"  EXTRAPOLATION_DIR: {EXTRAPOLATION_DIR}")
print(f"  RESULTS_DIR: {RESULTS_DIR}")
print(f"  MODELS_DIR: {MODELS_DIR}")

## Configuration

In [ ]:
# =============================================================================
# BEST HYPERPARAMETERS (loaded from tuning results)
# =============================================================================

# Load best hyperparameters from tuning results
BEST_CONFIG_PATH = BASE_PATH / "gSAGE" / "tuning" / "best_model_config.json"

with open(BEST_CONFIG_PATH, 'r') as f:
    best_config = json.load(f)

# Extract hyperparameters from the config file
BEST_HYPERPARAMS = {
    'num_layers': best_config['model_params']['num_layers'],
    'hidden_dim': best_config['model_params']['hidden_dim'],
    'learning_rate': best_config['training_params']['learning_rate'],
    'dropout': best_config['model_params']['dropout'],
    'aggr': best_config['model_params']['aggr'],
}

print(f"Loaded best hyperparameters from: {BEST_CONFIG_PATH}")
print(f"Config ID: {best_config['config_id']}")
print(f"Tuning performance: val_acc={best_config['performance']['val_accuracy']*100:.2f}%, test_acc={best_config['performance']['test_accuracy']*100:.2f}%")
print(f"\nHyperparameters:")
for k, v in BEST_HYPERPARAMS.items():
    print(f"  {k}: {v}")

In [ ]:
# =============================================================================
# TRAINING CONFIGURATION
# =============================================================================

# Total samples defining dataset scale (10^6)
TOTAL_SAMPLES = 1_000_000

# Training parameters
EPOCHS = 50
BATCH_SIZE = 256

# Rolling chunk configuration
# Dataset is split into NUM_CHUNKS chunks, each lasting CHUNK_LIFETIME epochs
# After each epoch, the oldest chunk is discarded and a new one is generated
NUM_CHUNKS = 4              # Dataset split into 4 chunks (25% each)
CHUNK_LIFETIME = 4          # Each chunk lasts 4 epochs before being discarded
VALIDATE_EVERY = 10         # run validation every N epochs

# Split ratios
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

# Total training samples (all chunks combined)
TRAIN_SAMPLES_TOTAL = int(TOTAL_SAMPLES * TRAIN_RATIO)

# Samples per chunk (25% of training data)
CHUNK_SIZE = TRAIN_SAMPLES_TOTAL // NUM_CHUNKS

# Eval set size built from cached baseline datasets (we only need val+test)
EVAL_TOTAL = int(TOTAL_SAMPLES * (VAL_RATIO + TEST_RATIO))

# Fixed parameters
IN_CHANNELS = 5  # Node features: [X?, Z?, d_North, d_West, d_time]
K_NEIGHBORS = 6  # For SparseGraph

# Error rate distribution for generated TRAINING data
P_VALUES = [0.001, 0.003, 0.005, 0.007]
P_WEIGHTS = [0.25, 0.25, 0.25, 0.25]

# Random seed for reproducibility
SEED = 42

print(f"Training Configuration:")
print(f"  Total samples: {TOTAL_SAMPLES:,}")
print(f"  Training samples total: {TRAIN_SAMPLES_TOTAL:,}")
print(f"  Eval total (val+test) samples: {EVAL_TOTAL:,}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"Rolling Chunk Configuration:")
print(f"  Number of chunks: {NUM_CHUNKS}")
print(f"  Chunk size: {CHUNK_SIZE:,} samples ({100/NUM_CHUNKS:.0f}% of training data)")
print(f"  Chunk lifetime: {CHUNK_LIFETIME} epochs")
print(f"  Validate every: {VALIDATE_EVERY} epochs")
print(f"  P values: {P_VALUES}")
print(f"  P weights: {P_WEIGHTS}")

In [ ]:
# =============================================================================
# SPLIT EXPERIMENT CONFIGURATIONS
# =============================================================================

SPLIT_EXPERIMENTS = {
    # Control
    'equal_333333': {'d3': 0.33, 'd5': 0.33, 'd7': 0.34},
    
    # Experiment A: Vary d=3 amount (d5:d7 = 1:1)
    'a1_d3_00': {'d3': 0.00, 'd5': 0.50, 'd7': 0.50},
    'a2_d3_10': {'d3': 0.10, 'd5': 0.45, 'd7': 0.45},
    'a3_d3_20': {'d3': 0.20, 'd5': 0.40, 'd7': 0.40},
    'a4_d3_40': {'d3': 0.40, 'd5': 0.30, 'd7': 0.30},
    'a5_d3_50': {'d3': 0.50, 'd5': 0.25, 'd7': 0.25},
    
    # Experiment B: Vary d5:d7 ratio (no d=3)
    'b1_d5heavy': {'d3': 0.00, 'd5': 0.80, 'd7': 0.20},
    'b2_d5more':  {'d3': 0.00, 'd5': 0.65, 'd7': 0.35},
    'b3_balanced': {'d3': 0.00, 'd5': 0.50, 'd7': 0.50},
    'b4_d7more':  {'d3': 0.00, 'd5': 0.35, 'd7': 0.65},
    'b5_d7heavy': {'d3': 0.00, 'd5': 0.20, 'd7': 0.80},
    
    # Experiment C: Extreme/boundary cases
    'c1_only_d3': {'d3': 1.00, 'd5': 0.00, 'd7': 0.00},
    'c2_only_d5': {'d3': 0.00, 'd5': 1.00, 'd7': 0.00},
    'c3_only_d7': {'d3': 0.00, 'd5': 0.00, 'd7': 1.00},
    'c4_no_d7':   {'d3': 0.50, 'd5': 0.50, 'd7': 0.00},
}

print("Split Experiments:")
print(f"{'Name':<25} {'d3':>6} {'d5':>6} {'d7':>6}")
print("-" * 50)
for name, config in SPLIT_EXPERIMENTS.items():
    hypothesis = config.get('hypothesis', 'N/A')
    print(f"{name:<25} {config['d3']*100:>5.0f}% {config['d5']*100:>5.0f}% {config['d7']*100:>5.0f}%")

In [ ]:
# =============================================================================
# SELECT WHICH EXPERIMENT TO RUN
# =============================================================================

# Worker assignments (each worker trains 5 experiments)
WORKER_EXPERIMENTS = {
    1: ['equal_333333', 'a1_d3_00', 'a2_d3_10', 'a3_d3_20', 'a4_d3_40'],
    2: ['a5_d3_50', 'b1_d5heavy', 'b2_d5more', 'b3_balanced', 'b4_d7more'],
    3: ['b5_d7heavy', 'c1_only_d3', 'c2_only_d5', 'c3_only_d7', 'c4_no_d7'],
}

# Select which worker to run (1, 2, or 3)
WORKER = 1  # Change this to 1, 2, or 3

if WORKER not in WORKER_EXPERIMENTS:
    raise ValueError(f"WORKER must be 1, 2, or 3, got {WORKER}")

experiments_to_run = WORKER_EXPERIMENTS[WORKER]
print(f"Worker {WORKER} will train {len(experiments_to_run)} experiments:")
for exp in experiments_to_run:
    print(f"  - {exp}")

## Helper Functions

In [ ]:
def generate_dataset_for_distance(d: int, n_samples: int, cache_name: str = None) -> list:
    """
    Generate or load a dataset for a specific distance.
    
    Args:
        d: Code distance
        n_samples: Number of samples to generate
        cache_name: Optional cache name to load from/save to
        
    Returns:
        List of PyG Data objects
    """
    cache = DatasetCache(base_path=BASE_PATH, device=device)
    
    # Try to load from cache first
    if cache_name:
        try:
            cache.load(cache_name, verbose=True)
            # Ensure we have enough samples
            if cache.size() >= n_samples:
                print(f"Loaded {n_samples:,} samples from cache '{cache_name}'")
                return cache.get_graphs(n=n_samples, shuffle=True)
            else:
                print(f"Cache has {cache.size():,} samples, need {n_samples:,}. Growing...")
                cache.ensure_size(n_samples, verbose=True)
                cache.save(cache_name)
                return cache.get_graphs(n=n_samples, shuffle=True)
        except FileNotFoundError:
            print(f"Cache '{cache_name}' not found. Generating new dataset...")
    
    # Generate new dataset
    cache.generate(
        d=d,
        n_samples=n_samples,
        p_values=P_VALUES,
        p_weights=P_WEIGHTS,
        k_neighbors=K_NEIGHBORS,
        verbose=True
    )
    
    # Save to cache if name provided
    if cache_name:
        cache.save(cache_name)
    
    return cache.get_graphs(shuffle=True)


def _mix_seed(base: int, scope: str, chunk_idx: int) -> int:
    """Deterministic seed mixer (uint32-ish)."""
    h = base
    for ch in str(scope):
        h = (h * 1000003 + ord(ch)) % (2**32 - 1)
    h = (h * 1009 + chunk_idx * 9176) % (2**32 - 1)
    return int(h)


def load_cached_eval_graphs_for_split(split_name: str, split_config: dict, total_eval: int) -> tuple:
    """Build fixed (val_graphs, test_graphs) from cached baseline datasets.

    We only materialize val+test graphs (default 200k), not the full 1e6.
    """
    # Determine counts by proportion (last distance gets remainder)
    items = [('d3', 3), ('d5', 5), ('d7', 7)]
    counts = []
    allocated = 0
    for key, d in items:
        prop = float(split_config.get(key, 0.0))
        n = int(total_eval * prop)
        counts.append((d, n))
        allocated += n
    # Add remainder to the last non-zero distance (or to d7 by default)
    remainder = total_eval - allocated
    if remainder != 0:
        for i in range(len(counts)-1, -1, -1):
            if split_config.get(f"d{counts[i][0]}", None) is not None:
                # counts tuple is (distance, n)
                d_i, n_i = counts[i]
                counts[i] = (d_i, n_i + remainder)
                break

    combined = []
    seed_i = _mix_seed(SEED, f"eval_{split_name}", 0)
    random.seed(seed_i)

    for d, n in counts:
        if n <= 0:
            continue
        cache_name = f"d{d}_baseline"
        cache = DatasetCache(base_path=BASE_PATH, device=device)
        cache.load(cache_name, verbose=True)
        if cache.size() < n:
            raise ValueError(f"Cached dataset '{cache_name}' has {cache.size():,} graphs, need {n:,} for eval")
        combined.extend(cache.get_graphs(n=n, shuffle=True))

    random.shuffle(combined)

    # Split eval into val/test equally (since VAL_RATIO == TEST_RATIO)
    half = len(combined) // 2
    val_graphs = combined[:half]
    test_graphs = combined[half:]

    print(f"Eval graphs built from cache for {split_name}: {len(val_graphs):,} val, {len(test_graphs):,} test")
    return val_graphs, test_graphs


# =============================================================================
# ROLLING CHUNK DATA GENERATION
# =============================================================================

def generate_chunk_for_split(split_name: str, split_config: dict, chunk_size: int, chunk_idx: int) -> list:
    """Generate a single chunk of training data for this split.
    
    Args:
        split_name: Name of the split experiment
        split_config: Dict with 'd3', 'd5', 'd7' proportions
        chunk_size: Number of samples in this chunk
        chunk_idx: Chunk index for deterministic seeding
        
    Returns:
        List of PyG Data objects
    """
    print(f"  [chunk {chunk_idx}] Generating {chunk_size:,} samples for {split_name}...")

    seed_i = _mix_seed(SEED, f"chunk_{split_name}", chunk_idx)
    random.seed(seed_i)
    np.random.seed(seed_i % (2**32 - 1))
    torch.manual_seed(seed_i)

    items = [('d3', 3), ('d5', 5), ('d7', 7)]
    counts = []
    allocated = 0
    for key, d in items:
        prop = float(split_config.get(key, 0.0))
        n = int(chunk_size * prop)
        counts.append((d, n))
        allocated += n
    remainder = chunk_size - allocated
    if remainder != 0:
        # put remainder into the largest-distance bucket by default (d7)
        for i in range(len(counts)-1, -1, -1):
            d_i, n_i = counts[i]
            if n_i > 0 or d_i == 7:
                counts[i] = (d_i, n_i + remainder)
                break

    combined = []
    for d, n in counts:
        if n <= 0:
            continue

        # Always generate training graphs (do not load/save cache)
        cache = DatasetCache(base_path=BASE_PATH, device=device)
        cache.generate(
            d=d,
            n_samples=n,
            p_values=P_VALUES,
            p_weights=P_WEIGHTS,
            k_neighbors=K_NEIGHBORS,
            verbose=False,
        )
        combined.extend(cache.get_graphs(shuffle=True))
        del cache

    random.shuffle(combined)
    return combined


def get_active_chunk_indices(epoch: int, num_chunks: int = NUM_CHUNKS) -> list:
    """Return list of chunk indices that should be active at given epoch.
    
    At epoch E, chunks [E, E+1, E+2, E+3] are active (for NUM_CHUNKS=4).
    """
    return list(range(epoch, epoch + num_chunks))


def initialize_chunks(split_name: str, split_config: dict, start_epoch: int, 
                      chunk_size: int = CHUNK_SIZE, num_chunks: int = NUM_CHUNKS) -> dict:
    """Initialize all chunks needed for the starting epoch.
    
    Args:
        split_name: Name of the split experiment
        split_config: Dict with 'd3', 'd5', 'd7' proportions
        start_epoch: Starting epoch (0-indexed)
        chunk_size: Samples per chunk
        num_chunks: Number of chunks in dataset
        
    Returns:
        Dict mapping chunk_idx -> list of graphs
    """
    print(f"\n[Rolling chunks] Initializing {num_chunks} chunks for epoch {start_epoch}...")
    chunks = {}
    for chunk_idx in get_active_chunk_indices(start_epoch, num_chunks):
        chunks[chunk_idx] = generate_chunk_for_split(split_name, split_config, chunk_size, chunk_idx)
    total_samples = sum(len(c) for c in chunks.values())
    print(f"[Rolling chunks] Initialized: {len(chunks)} chunks, {total_samples:,} total samples")
    return chunks


def roll_chunks(chunks: dict, split_name: str, split_config: dict, new_epoch: int,
                chunk_size: int = CHUNK_SIZE, num_chunks: int = NUM_CHUNKS) -> dict:
    """Roll the chunk window: discard oldest chunk, generate new chunk.
    
    Called BEFORE training on new_epoch. Discards chunk that expired and
    generates the new chunk needed for this epoch.
    
    Args:
        chunks: Current dict of chunk_idx -> graphs
        split_name: Name of the split experiment
        split_config: Dict with 'd3', 'd5', 'd7' proportions
        new_epoch: The epoch we're about to train on
        chunk_size: Samples per chunk
        num_chunks: Number of chunks in dataset
        
    Returns:
        Updated chunks dict
    """
    import gc
    
    # The oldest chunk to discard is from (new_epoch - 1) - (num_chunks - 1) = new_epoch - num_chunks
    oldest_idx = new_epoch - num_chunks
    if oldest_idx in chunks:
        print(f"\n[Rolling chunks] Epoch {new_epoch}: discarding chunk {oldest_idx}, ", end="")
        del chunks[oldest_idx]
        gc.collect()
    
    # Generate new chunk: the newest active chunk is new_epoch + num_chunks - 1
    new_idx = new_epoch + num_chunks - 1
    if new_idx not in chunks:
        print(f"generating chunk {new_idx}...")
        chunks[new_idx] = generate_chunk_for_split(split_name, split_config, chunk_size, new_idx)
    
    return chunks


def combine_chunks(chunks: dict) -> list:
    """Combine all chunks into a single list of graphs, shuffled."""
    combined = []
    for chunk_idx in sorted(chunks.keys()):
        combined.extend(chunks[chunk_idx])
    random.shuffle(combined)
    return combined


# Legacy function for backward compatibility (used in final evaluation)
def generate_fresh_training_graphs_for_split(split_name: str, split_config: dict, total_train: int, block_idx: int) -> list:
    """Generate a fresh TRAINING dataset for this split (no disk cache).
    
    Legacy function kept for backward compatibility with final evaluation.
    """
    print(f"\n[train eval] {split_name}: generating evaluation graphs (total={total_train:,})")

    seed_i = _mix_seed(SEED, f"train_{split_name}", block_idx)
    random.seed(seed_i)
    np.random.seed(seed_i % (2**32 - 1))
    torch.manual_seed(seed_i)

    items = [('d3', 3), ('d5', 5), ('d7', 7)]
    counts = []
    allocated = 0
    for key, d in items:
        prop = float(split_config.get(key, 0.0))
        n = int(total_train * prop)
        counts.append((d, n))
        allocated += n
    remainder = total_train - allocated
    if remainder != 0:
        for i in range(len(counts)-1, -1, -1):
            d_i, n_i = counts[i]
            if n_i > 0 or d_i == 7:
                counts[i] = (d_i, n_i + remainder)
                break

    combined = []
    for d, n in counts:
        if n <= 0:
            continue

        cache = DatasetCache(base_path=BASE_PATH, device=device)
        cache.generate(
            d=d,
            n_samples=n,
            p_values=P_VALUES,
            p_weights=P_WEIGHTS,
            k_neighbors=K_NEIGHBORS,
            verbose=True,
        )
        combined.extend(cache.get_graphs(shuffle=True))
        del cache

    random.shuffle(combined)
    print(f"Evaluation graphs ready: {len(combined):,} samples")
    return combined

In [ ]:
def load_mixed_dataset(split_config: dict, total_samples: int) -> list:
    """
    Load and combine datasets according to split configuration.
    
    Args:
        split_config: Dict with 'd3', 'd5', 'd7' keys specifying proportions
        total_samples: Total number of samples in combined dataset
        
    Returns:
        Combined list of PyG Data objects, shuffled
    """
    combined_graphs = []
    
    for distance_key, proportion in [('d3', 3), ('d5', 5), ('d7', 7)]:
        if split_config[distance_key] > 0:
            n_samples = int(total_samples * split_config[distance_key])
            if n_samples > 0:
                cache_name = f"d{proportion}_baseline"
                print(f"\nLoading d={proportion} dataset ({n_samples:,} samples, {split_config[distance_key]*100:.0f}%)...")
                graphs = generate_dataset_for_distance(proportion, n_samples, cache_name)
                combined_graphs.extend(graphs)
                print(f"  Added {len(graphs):,} d={proportion} samples")
    
    # Shuffle combined dataset
    random.seed(SEED)
    random.shuffle(combined_graphs)
    
    print(f"\nCombined dataset: {len(combined_graphs):,} total samples")
    return combined_graphs

In [ ]:
def split_train_val_test(graphs: list, train_ratio: float = 0.8, val_ratio: float = 0.1, seed: int = 42):
    """
    Split dataset into train/val/test sets.
    
    Args:
        graphs: List of PyG Data objects
        train_ratio: Proportion for training
        val_ratio: Proportion for validation
        seed: Random seed
        
    Returns:
        Tuple of (train_graphs, val_graphs, test_graphs)
    """
    random.seed(seed)
    indices = list(range(len(graphs)))
    random.shuffle(indices)
    
    n = len(graphs)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    
    train_idx = indices[:train_end]
    val_idx = indices[train_end:val_end]
    test_idx = indices[val_end:]
    
    train_graphs = [graphs[i] for i in train_idx]
    val_graphs = [graphs[i] for i in val_idx]
    test_graphs = [graphs[i] for i in test_idx]
    
    print(f"Dataset split: {len(train_graphs):,} train, {len(val_graphs):,} val, {len(test_graphs):,} test")
    
    return train_graphs, val_graphs, test_graphs

In [ ]:
def evaluate_model(model, graphs, device, batch_size=256):
    """
    Evaluate model accuracy on a set of graphs.
    
    Args:
        model: GraphSAGE model wrapper
        graphs: List of PyG Data objects
        device: Torch device
        batch_size: Batch size for evaluation
        
    Returns:
        Accuracy as a float
    """
    model.model.eval()
    loader = DataLoader(graphs, batch_size=batch_size, shuffle=False)
    
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = model.model(batch)
            y = batch.y.float().view(-1, 1)
            correct += ((pred > 0.5).float() == y).sum().item()
            total += y.size(0)
    
    return correct / total if total > 0 else 0.0

In [ ]:
def save_model(model, split_name: str, metrics: dict):
    """
    Save trained model with metadata.
    
    Args:
        model: GraphSAGE model wrapper
        split_name: Name of the split experiment
        metrics: Dict with training metrics
    """
    model_path = MODELS_DIR / f"{split_name}.pt"
    
    save_dict = {
        'state_dict': model.model.state_dict(),
        'config': model._config,
        'split_name': split_name,
        'split_config': SPLIT_EXPERIMENTS[split_name],
        'hyperparams': BEST_HYPERPARAMS,
        'metrics': metrics,
        'timestamp': datetime.now().isoformat(),
    }
    
    torch.save(save_dict, model_path)
    print(f"Model saved to: {model_path}")
    return model_path

In [ ]:
def save_training_results(split_name: str, epoch_metrics: dict, final_metrics: dict = None):
    """Save training results to JSON with full epoch-by-epoch metrics."""
    results_path = RESULTS_DIR / f"{split_name}_training.json"

    results = {
        'split_name': split_name,
        'split_config': SPLIT_EXPERIMENTS[split_name],
        'hyperparams': BEST_HYPERPARAMS,
        'training_config': {
            'total_samples': TOTAL_SAMPLES,
            'epochs': EPOCHS,
            'batch_size': BATCH_SIZE,
            'num_chunks': NUM_CHUNKS,
            'chunk_size': CHUNK_SIZE,
            'chunk_lifetime': CHUNK_LIFETIME,
            'validate_every': VALIDATE_EVERY,
            'train_samples_total': TRAIN_SAMPLES_TOTAL,
            'eval_total': EVAL_TOTAL,
            'p_values': P_VALUES,
            'p_weights': P_WEIGHTS,
        },
        'epoch_metrics': {
            'epochs': epoch_metrics.get('epoch', []),
            'train_loss': epoch_metrics.get('train_loss', []),
            'train_accuracy': epoch_metrics.get('train_accuracy', []),
            'val_accuracy': epoch_metrics.get('val_accuracy', []),
            'epoch_time_seconds': epoch_metrics.get('epoch_time_seconds', []),
            'learning_rate': epoch_metrics.get('learning_rate', []),
            'gpu_memory_mb': epoch_metrics.get('gpu_memory_mb', []),
            'active_chunks': epoch_metrics.get('active_chunks', []),
            'train_samples': epoch_metrics.get('train_samples', []),
        },
        'final_metrics': final_metrics if final_metrics is not None else {},
        'timestamp': datetime.now().isoformat(),
    }

    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)

    print(f"Results saved to: {results_path}")


# =============================================================================
# CHECKPOINTING (per split)
# =============================================================================

def _checkpoint_path(split_name: str) -> Path:
    return MODELS_DIR / f"{split_name}_checkpoint.pt"


def load_checkpoint(split_name: str) -> dict:
    path = _checkpoint_path(split_name)
    if not path.exists():
        return None
    print(f"Loading checkpoint: {path}")
    return torch.load(path, map_location=device, weights_only=False)


def save_checkpoint(split_name: str, checkpoint: dict):
    path = _checkpoint_path(split_name)
    torch.save(checkpoint, path)


# =============================================================================
# TRAINING LOOP (rolling chunk refresh)
# =============================================================================

def train_with_rolling_chunks(
    split_name: str,
    split_config: dict,
    model: GraphSAGE,
    val_graphs: list,
    epochs: int,
    batch_size: int,
    lr: float,
    validate_every: int,
    start_epoch: int = 0,
    initial_metrics: dict = None,
    optimizer_state_dict: dict = None,
    initial_chunk_indices: list = None,
    verbose: bool = True,
):
    """Train model with rolling chunk data refresh.
    
    Instead of replacing the entire dataset every N epochs, we maintain
    NUM_CHUNKS chunks where each chunk lasts CHUNK_LIFETIME epochs.
    After each epoch, the oldest chunk is discarded and a new one is generated.
    
    Args:
        split_name: Name of the split experiment
        split_config: Dict with 'd3', 'd5', 'd7' proportions
        model: GraphSAGE model wrapper
        val_graphs: Validation graphs
        epochs: Total number of epochs
        batch_size: Batch size
        lr: Learning rate
        validate_every: Validate every N epochs
        start_epoch: Starting epoch (for resumption)
        initial_metrics: Pre-existing metrics if resuming
        optimizer_state_dict: Optimizer state to restore if resuming
        initial_chunk_indices: Chunk indices to regenerate if resuming
        verbose: Whether to print progress
    """
    import gc

    optimizer = torch.optim.Adam(model.model.parameters(), lr=lr)
    loss_fn = torch.nn.BCELoss()

    if optimizer_state_dict is not None:
        optimizer.load_state_dict(optimizer_state_dict)
        if verbose:
            print("Restored optimizer state from checkpoint")

    if initial_metrics is not None:
        epoch_metrics = initial_metrics.copy()
    else:
        epoch_metrics = {
            'epoch': [],
            'train_loss': [],
            'train_accuracy': [],
            'val_accuracy': [],
            'epoch_time_seconds': [],
            'active_chunks': [],
            'learning_rate': [],
            'gpu_memory_mb': [],
            'train_samples': [],
        }

    total_start_time = time.time()

    # Initialize chunks for starting epoch
    # If resuming, we regenerate the chunks that were active
    if initial_chunk_indices is not None and len(initial_chunk_indices) > 0:
        print(f"\n[Rolling chunks] Regenerating chunks from checkpoint: {initial_chunk_indices}")
        chunks = {}
        for chunk_idx in initial_chunk_indices:
            chunks[chunk_idx] = generate_chunk_for_split(split_name, split_config, CHUNK_SIZE, chunk_idx)
    else:
        chunks = initialize_chunks(split_name, split_config, start_epoch)
    
    train_graphs = combine_chunks(chunks)
    loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=True)

    for epoch in range(start_epoch, epochs):
        epoch_start = time.time()

        # Roll chunks if not the first epoch
        if epoch > start_epoch:
            chunks = roll_chunks(chunks, split_name, split_config, epoch)
            train_graphs = combine_chunks(chunks)
            loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=True)

        model.model.train()
        epoch_loss = 0.0
        epoch_correct = 0
        epoch_total = 0

        active_chunk_list = sorted(chunks.keys())
        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs} [chunks {active_chunk_list}]", 
                    disable=not verbose, leave=False)
        for batch in pbar:
            batch = batch.to(device)
            pred = model.model(batch)
            y = batch.y.float().view(-1, 1)

            loss = loss_fn(pred, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            epoch_correct += ((pred > 0.5).float() == y).sum().item()
            epoch_total += y.size(0)

            current_acc = 100.0 * epoch_correct / epoch_total
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{current_acc:.1f}%'})

        pbar.close()

        avg_train_loss = epoch_loss / max(1, len(loader))
        train_acc = epoch_correct / max(1, epoch_total)

        do_validate = ((epoch + 1) % validate_every == 0) or (epoch + 1 == epochs)
        if do_validate:
            val_acc = evaluate_model(model, val_graphs, device, batch_size=batch_size)
        else:
            val_acc = float('nan')

        epoch_time = time.time() - epoch_start

        # GPU memory tracking
        if torch.cuda.is_available():
            gpu_memory_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)
            torch.cuda.reset_peak_memory_stats()
        else:
            gpu_memory_mb = 0.0

        # Learning rate from optimizer
        current_lr = optimizer.param_groups[0]['lr']

        # Training samples
        train_samples_count = len(train_graphs)

        epoch_metrics['epoch'].append(epoch + 1)
        epoch_metrics['train_loss'].append(avg_train_loss)
        epoch_metrics['train_accuracy'].append(train_acc)
        epoch_metrics['val_accuracy'].append(val_acc)
        epoch_metrics['epoch_time_seconds'].append(epoch_time)
        epoch_metrics['active_chunks'].append(active_chunk_list)
        epoch_metrics['learning_rate'].append(current_lr)
        epoch_metrics['gpu_memory_mb'].append(gpu_memory_mb)
        epoch_metrics['train_samples'].append(train_samples_count)

        if verbose:
            val_str = f"{val_acc*100:.2f}%" if not np.isnan(val_acc) else "--"
            print(f"Epoch {epoch+1}/{epochs} | Chunks: {active_chunk_list} | "
                  f"Loss: {avg_train_loss:.4f} | Train: {train_acc*100:.2f}% | Val: {val_str}")

        # checkpoint every epoch - save chunk indices for regeneration on resume
        save_checkpoint(split_name, {
            'state_dict': model.model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'current_epoch': epoch + 1,
            'epoch_metrics': epoch_metrics,
            'validate_every': validate_every,
            'active_chunk_indices': active_chunk_list,
            'timestamp': datetime.now().isoformat(),
        })

    total_time = time.time() - total_start_time
    epoch_metrics['total_training_time_seconds'] = total_time
    epoch_metrics['_optimizer_state_dict'] = optimizer.state_dict()

    return epoch_metrics

## Training Pipeline

In [ ]:
def run_split_experiment(split_name: str, split_config: dict):
    """Run one extrapolation split with rolling chunk training + cached eval."""
    print(f"\n{'='*70}")
    print(f"EXPERIMENT: {split_name}")
    print(f"{'='*70}")
    print(f"Split: d3={split_config['d3']*100:.0f}%, d5={split_config['d5']*100:.0f}%, d7={split_config['d7']*100:.0f}%")
    print(f"Rolling chunks: {NUM_CHUNKS} chunks x {CHUNK_SIZE:,} samples, {CHUNK_LIFETIME} epoch lifetime")
    hypothesis = split_config.get('hypothesis', 'N/A')
    if hypothesis != 'N/A':
        print(f"Hypothesis: {hypothesis}")
    print(f"{'='*70}")

    start_time = time.time()

    # 1) Cached EVAL (val/test) built from baseline caches
    print("\n[1/4] Building cached eval datasets (val/test)...")
    val_graphs, test_graphs = load_cached_eval_graphs_for_split(split_name, split_config, total_eval=EVAL_TOTAL)

    # 2) Initialize model
    print("\n[2/4] Initializing model...")
    model = GraphSAGE(
        nickname=f"extrapolation_{split_name}",
        in_channels=IN_CHANNELS,
        hidden_dim=BEST_HYPERPARAMS['hidden_dim'],
        num_layers=BEST_HYPERPARAMS['num_layers'],
        dropout=BEST_HYPERPARAMS['dropout'],
        aggr=BEST_HYPERPARAMS['aggr'],
        device=device,
        base_path=BASE_PATH,
        seed=SEED
    )

    # 3) Resume if checkpoint exists
    checkpoint = load_checkpoint(split_name)
    start_epoch = 0
    initial_metrics = None
    optimizer_state_dict = None
    initial_chunk_indices = None

    if checkpoint is not None:
        start_epoch = int(checkpoint.get('current_epoch', 0))
        initial_metrics = checkpoint.get('epoch_metrics', None)
        optimizer_state_dict = checkpoint.get('optimizer_state_dict', None)
        initial_chunk_indices = checkpoint.get('active_chunk_indices', None)
        if 'state_dict' in checkpoint:
            model.model.load_state_dict(checkpoint['state_dict'])
        if start_epoch >= EPOCHS:
            print(f"Checkpoint indicates training complete at epoch {start_epoch}. Skipping training.")

    # 4) Train with rolling chunks
    print(f"\n[3/4] Training model (rolling chunks: 25% replaced each epoch)...")

    if start_epoch < EPOCHS:
        epoch_metrics = train_with_rolling_chunks(
            split_name=split_name,
            split_config=split_config,
            model=model,
            val_graphs=val_graphs,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            lr=BEST_HYPERPARAMS['learning_rate'],
            validate_every=VALIDATE_EVERY,
            start_epoch=start_epoch,
            initial_metrics=initial_metrics,
            optimizer_state_dict=optimizer_state_dict,
            initial_chunk_indices=initial_chunk_indices,
            verbose=True,
        )
    else:
        epoch_metrics = initial_metrics
        if epoch_metrics is None:
            epoch_metrics = {
                'epoch': [],
                'train_loss': [],
                'train_accuracy': [],
                'val_accuracy': [],
                'epoch_time_seconds': [],
                'active_chunks': [],
                'learning_rate': [],
                'gpu_memory_mb': [],
                'train_samples': [],
            }

    # Evaluate
    print("\n[4/4] Evaluating model...")
    # Generate fresh evaluation data (using chunk index = EPOCHS for fresh data)
    train_eval_subset = generate_fresh_training_graphs_for_split(split_name, split_config, total_train=10_000, block_idx=EPOCHS)

    train_acc = evaluate_model(model, train_eval_subset, device, batch_size=BATCH_SIZE)
    val_acc = evaluate_model(model, val_graphs, device, batch_size=BATCH_SIZE)
    test_acc = evaluate_model(model, test_graphs, device, batch_size=BATCH_SIZE)

    training_time = time.time() - start_time

    epoch_losses = epoch_metrics.get('train_loss', [])
    final_loss = epoch_losses[-1] if epoch_losses else None

    # Prepare final metrics for model save (backward compatibility)
    metrics = {
        'epoch_losses': epoch_losses,
        'final_loss': final_loss,
        'train_accuracy': train_acc,
        'val_accuracy': val_acc,
        'test_accuracy': test_acc,
        'training_time_seconds': training_time,
    }

    # Prepare final_metrics for results JSON
    final_metrics = {
        'final_loss': final_loss,
        'train_accuracy': train_acc,
        'val_accuracy': val_acc,
        'test_accuracy': test_acc,
        'total_training_time_seconds': training_time,
    }

    save_model(model, split_name, metrics)
    save_training_results(split_name, epoch_metrics, final_metrics)

    print(f"\n{'='*70}")
    print(f"EXPERIMENT COMPLETE: {split_name}")
    print(f"{'='*70}")
    print(f"  Training time: {training_time/60:.1f} minutes")
    if final_loss is not None:
        print(f"  Final loss: {final_loss:.4f}")
    print(f"  Train accuracy: {train_acc*100:.2f}%")
    print(f"  Val accuracy: {val_acc*100:.2f}%")
    print(f"  Test accuracy: {test_acc*100:.2f}%")
    print(f"{'='*70}")

    # Clean up memory
    del model, val_graphs, test_graphs, train_eval_subset
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    return metrics

## Run Training Experiments

In [ ]:
# Confirm hyperparameters loaded successfully
print(f"Using hyperparameters from config {best_config['config_id']}:")
for k, v in BEST_HYPERPARAMS.items():
    print(f"  {k}: {v}")

print(f"\n{'#'*70}")
print(f"STARTING EXTRAPOLATION TRAINING EXPERIMENTS")
print(f"{'#'*70}")
print(f"Experiments to run: {len(experiments_to_run)}")
for exp in experiments_to_run:
    print(f"  - {exp}")

In [ ]:
# Run all selected experiments
all_results = {}

for i, split_name in enumerate(experiments_to_run):
    print(f"\n\n{'#'*70}")
    print(f"EXPERIMENT {i+1}/{len(experiments_to_run)}: {split_name}")
    print(f"{'#'*70}")
    
    split_config = SPLIT_EXPERIMENTS[split_name]
    metrics = run_split_experiment(split_name, split_config)
    all_results[split_name] = metrics

print(f"\n\n{'#'*70}")
print(f"ALL EXPERIMENTS COMPLETE")
print(f"{'#'*70}")

## Results Summary

In [ ]:
# Create summary table
if all_results:
    summary_data = []
    for split_name, metrics in all_results.items():
        config = SPLIT_EXPERIMENTS[split_name]
        summary_data.append({
            'Split': split_name,
            'd3 %': config['d3'] * 100,
            'd5 %': config['d5'] * 100,
            'd7 %': config['d7'] * 100,
            'Train Acc': metrics['train_accuracy'] * 100,
            'Val Acc': metrics['val_accuracy'] * 100,
            'Test Acc': metrics['test_accuracy'] * 100,
            'Final Loss': metrics['final_loss'],
            'Time (min)': metrics['training_time_seconds'] / 60,
        })
    
    df_summary = pd.DataFrame(summary_data)
    print("\nTraining Results Summary:")
    print(df_summary.to_string(index=False))
    
    # Save summary
    summary_path = RESULTS_DIR / "training_summary.csv"
    df_summary.to_csv(summary_path, index=False)
    print(f"\nSummary saved to: {summary_path}")

In [ ]:
# Plot training curves
if all_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss curves
    ax1 = axes[0]
    for split_name, metrics in all_results.items():
        ax1.plot(metrics['epoch_losses'], label=split_name)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss by Split Configuration')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Accuracy comparison
    ax2 = axes[1]
    split_names = list(all_results.keys())
    val_accs = [all_results[s]['val_accuracy'] * 100 for s in split_names]
    test_accs = [all_results[s]['test_accuracy'] * 100 for s in split_names]
    
    x = np.arange(len(split_names))
    width = 0.35
    ax2.bar(x - width/2, val_accs, width, label='Validation')
    ax2.bar(x + width/2, test_accs, width, label='Test')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Accuracy by Split Configuration')
    ax2.set_xticks(x)
    ax2.set_xticklabels(split_names, rotation=45, ha='right')
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    
    # Save plot
    plot_path = PLOTS_DIR / "training_results.png"
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f"Plot saved to: {plot_path}")
    
    plt.show()

## Next Steps

After training completes:
1. Review the training results summary above
2. Run `testing.ipynb` to evaluate extrapolation to d=9 and d=11
3. Compare how different split configurations affect extrapolation performance

In [ ]:
from google.colab import runtime
runtime.unassign()